In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
TRAIN_FILE = "/content/drive/MyDrive/MATH 5470 new/home-credit-default-risk/train_app_categorical_pca.csv"
TEST_FILE = "/content/drive/MyDrive/MATH 5470 new/home-credit-default-risk/test_app_categorical_pca .csv"

ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"
RANDOM_STATE = 42
VALIDATION_SIZE = 0.2  # 80/20 split for validation

# Baseline RandomForest parameters
BASE_RF_PARAMS = dict(
    n_estimators=250,
    min_samples_split=16,
    min_samples_leaf=18,
    random_state=0,
    n_jobs=-1,
)

# Hyper-parameter configurations to try (maximum three)
RF_PARAM_CONFIGS = [
    {"n_estimators": 200, "min_samples_split": 16, "min_samples_leaf": 15},
    {"n_estimators": 250, "min_samples_split": 14, "min_samples_leaf": 18},
    {"n_estimators": 300, "min_samples_split": 18, "min_samples_leaf": 20},
]

# ---------------------------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------------------------
def preprocess_data(app_train, app_test):
    """
    Preprocess the Home Credit Default Risk data
    """
    print("\nPreprocessing data...")

    # Separate target variable from training data
    if 'TARGET' not in app_train.columns:
        print("Error: 'TARGET' column not found in training data")
        if app_train.shape[1] > 1:
            y = app_train.iloc[:, 1]  # Assuming second column is target
            X = app_train.drop(app_train.columns[1], axis=1)
            print("Using second column as target variable")
        else:
            raise ValueError("Cannot find target variable in training data")
    else:
        y = app_train['TARGET']
        X = app_train.drop('TARGET', axis=1)

    test_df = app_test.copy()

    # Store test IDs for submission
    if 'SK_ID_CURR' in test_df.columns:
        test_ids = test_df['SK_ID_CURR']
    else:
        test_ids = pd.Series(range(1, len(test_df) + 1))
        print("Created sequential test IDs")

    # Remove identifier columns
    id_cols = ['SK_ID_CURR']
    X = X.drop(id_cols, axis=1, errors='ignore')
    test_df = test_df.drop(id_cols, axis=1, errors='ignore')

    # Check for missing values
    missing_train = X.isnull().sum().sum()
    missing_test = test_df.isnull().sum().sum()

    print(f"Missing values in training: {missing_train}")
    print(f"Missing values in test: {missing_test}")

    # Fill any remaining missing values with 0
    if missing_train > 0:
        X = X.fillna(0)
        print("Filled missing values in training data with 0")

    if missing_test > 0:
        test_df = test_df.fillna(0)
        print("Filled missing values in test data with 0")

    # Check for infinite values
    inf_train = np.isinf(X).sum().sum()
    inf_test = np.isinf(test_df).sum().sum()

    if inf_train > 0:
        print(f"Found {inf_train} infinite values in training data, replacing with max/min values")
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    if inf_test > 0:
        print(f"Found {inf_test} infinite values in test data, replacing with max/min values")
        test_df = test_df.replace([np.inf, -np.inf], np.nan).fillna(0)

    print(f"Features after preprocessing: {X.shape[1]}")
    print(f"Training set shape: {X.shape}")
    print(f"Test set shape: {test_df.shape}")

    return X, y, test_df, test_ids

# ---------------------------------------------------------------------------
# Main training routine
# ---------------------------------------------------------------------------
def main():
    # Load data
    train_df = pd.read_csv(TRAIN_FILE)
    test_df = pd.read_csv(TEST_FILE)

    # Preprocess using the provided function
    X, y, X_test, test_ids = preprocess_data(train_df, test_df)

    # Train/validation split
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=VALIDATION_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    # Evaluate each hyper-parameter configuration
    results = []
    for idx, param_delta in enumerate(RF_PARAM_CONFIGS, start=1):
        rf_params = BASE_RF_PARAMS.copy()
        rf_params.update(param_delta)
        model = RandomForestClassifier(**rf_params)

        print(f"\n=== Configuration {idx} ===")
        print("Parameters:", rf_params)

        model.fit(X_train, y_train)

        y_valid_pred = model.predict(X_valid)
        y_valid_proba = model.predict_proba(X_valid)[:, 1]

        val_accuracy = accuracy_score(y_valid, y_valid_pred)
        val_auc = roc_auc_score(y_valid, y_valid_proba)
        val_report = classification_report(y_valid, y_valid_pred, digits=4)

        print("Validation Accuracy:", round(val_accuracy, 4))
        print("Validation ROC AUC:", round(val_auc, 4))
        print("\nClassification Report:\n", val_report)

        results.append(
            {
                "config_index": idx,
                "params": rf_params,
                "model": model,
                "accuracy": val_accuracy,
                "roc_auc": val_auc,
                "report": val_report,
            }
        )

    # Select the best model by validation ROC AUC
    best_result = max(results, key=lambda x: x["roc_auc"])
    best_params = best_result["params"]
    best_auc = best_result["roc_auc"]

    print("\n=== Best Configuration Selected ===")
    print("Parameters:", best_params)
    print("Validation ROC AUC:", round(best_auc, 4))

    # Retrain best model on the full training data
    best_model = RandomForestClassifier(**best_params)
    best_model.fit(X, y)

    # Test predictions
    test_pred_proba = best_model.predict_proba(X_test)[:, 1]
    test_pred_label = best_model.predict(X_test)

    submission = pd.DataFrame(
        {
            ID_COLUMN: test_ids.values,
            "TARGET": test_pred_proba,

        }
    )

    output_file = Path("rf_predictions.csv")
    submission.to_csv(output_file, index=False)
    print(f"\nTest predictions saved to: {output_file.resolve()}")

    return {
        "best_params": best_params,
        "validation_results": results,
        "submission_path": output_file,
    }


if __name__ == "__main__":
    main()


Preprocessing data...
Missing values in training: 0
Missing values in test: 0
Features after preprocessing: 123
Training set shape: (307511, 123)
Test set shape: (48744, 123)

=== Configuration 1 ===
Parameters: {'n_estimators': 200, 'min_samples_split': 16, 'min_samples_leaf': 15, 'random_state': 0, 'n_jobs': -1}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation Accuracy: 0.9193
Validation ROC AUC: 0.7414

Classification Report:
               precision    recall  f1-score   support

           0     0.9193    1.0000    0.9579     56538
           1     0.0000    0.0000    0.0000      4965

    accuracy                         0.9193     61503
   macro avg     0.4596    0.5000    0.4790     61503
weighted avg     0.8451    0.9193    0.8806     61503


=== Configuration 2 ===
Parameters: {'n_estimators': 250, 'min_samples_split': 14, 'min_samples_leaf': 18, 'random_state': 0, 'n_jobs': -1}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation Accuracy: 0.9193
Validation ROC AUC: 0.7429

Classification Report:
               precision    recall  f1-score   support

           0     0.9193    1.0000    0.9579     56538
           1     0.0000    0.0000    0.0000      4965

    accuracy                         0.9193     61503
   macro avg     0.4596    0.5000    0.4790     61503
weighted avg     0.8451    0.9193    0.8806     61503


=== Configuration 3 ===
Parameters: {'n_estimators': 300, 'min_samples_split': 18, 'min_samples_leaf': 20, 'random_state': 0, 'n_jobs': -1}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation Accuracy: 0.9193
Validation ROC AUC: 0.7441

Classification Report:
               precision    recall  f1-score   support

           0     0.9193    1.0000    0.9579     56538
           1     0.0000    0.0000    0.0000      4965

    accuracy                         0.9193     61503
   macro avg     0.4596    0.5000    0.4790     61503
weighted avg     0.8451    0.9193    0.8806     61503


=== Best Configuration Selected ===
Parameters: {'n_estimators': 300, 'min_samples_split': 18, 'min_samples_leaf': 20, 'random_state': 0, 'n_jobs': -1}
Validation ROC AUC: 0.7441

Test predictions saved to: /content/rf_predictions.csv
